# Task6 v2: Gene-Adaptive PPI Visualization

对比 4 种方法: **Baseline → PPI (v1) → Adaptive PPI (v2)**  
新增: Gate 分析 / Morphology-Informativeness 分层 / Expression Profile

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import pearsonr, spearmanr, wilcoxon
import pandas as pd
import json
import os

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'pdf.fonttype': 42,
})

FIGDIR = '../figures/v2/'
os.makedirs(FIGDIR, exist_ok=True)
print('Figures will be saved to:', os.path.abspath(FIGDIR))

## 1. 配置路径

根据环境选择 HPC 或 本地 路径。

In [ ]:
# ====== 根据环境修改 ======
if os.path.exists('/gpfsdata/home/renyixiang'):
    # HPC
    MG_RESULT_DIR = '/gpfsdata/home/renyixiang/YuanLab/data/MG_Xenium/SpatialEx_results_wgcna_ppi_MG/'
    SKIN_RESULT_DIR = '/gpfsdata/home/renyixiang/YuanLab/data/Skin_Xenium/SpatialEx_results_wgcna_ppi_skin/'
else:
    # Local (modify if needed)
    MG_RESULT_DIR = './SpatialEx_results_wgcna_ppi_MG/'
    SKIN_RESULT_DIR = './SpatialEx_results_wgcna_ppi_skin/'

CONFIGS = ['0_baseline', '3_all_three', '4_adaptive_ppi']
CONFIG_LABELS = ['Baseline', 'PPI (v1)', 'Adaptive PPI (v2)']
CONFIG_COLORS = ['#999999', '#4DBBD5', '#E64B35']

print('MG dir:', MG_RESULT_DIR)
print('Skin dir:', SKIN_RESULT_DIR)
for c in CONFIGS:
    mg_ok = os.path.exists(os.path.join(MG_RESULT_DIR, c, 'metrics.json'))
    skin_ok = os.path.exists(os.path.join(SKIN_RESULT_DIR, c, 'metrics.json'))
    print(f'  {c}: MG={"OK" if mg_ok else "MISSING"}, Skin={"OK" if skin_ok else "MISSING"}')

## 2. 加载 Metrics

In [ ]:
def load_all_metrics(result_dir, configs):
    """加载所有 config 的 metrics.json 和 per-gene arrays。"""
    data = {}
    for cfg in configs:
        cfg_dir = os.path.join(result_dir, cfg)
        metrics_path = os.path.join(cfg_dir, 'metrics.json')
        if not os.path.exists(metrics_path):
            print(f'  WARNING: {cfg} metrics.json not found, skipping')
            continue
        with open(metrics_path) as f:
            m = json.load(f)
        
        entry = {'metrics': m}
        # Load per-gene arrays
        for s in [1, 2]:
            for metric in ['pcc', 'ssim', 'morans', 'spcc']:
                arr_path = os.path.join(cfg_dir, f'slice{s}_{metric}_per_gene.npy')
                if os.path.exists(arr_path):
                    entry[f'{metric}_s{s}'] = np.load(arr_path)
        
        # Load gate values (v2 only)
        for s in [1, 2]:
            gate_path = os.path.join(cfg_dir, f'slice{s}_gate_values.npy')
            if os.path.exists(gate_path):
                entry[f'gate_s{s}'] = np.load(gate_path)
            gate_json = os.path.join(cfg_dir, f'slice{s}_gate_summary.json')
            if os.path.exists(gate_json):
                with open(gate_json) as f:
                    entry[f'gate_summary_s{s}'] = json.load(f)
        
        data[cfg] = entry
    return data

mg_data = load_all_metrics(MG_RESULT_DIR, CONFIGS)
skin_data = load_all_metrics(SKIN_RESULT_DIR, CONFIGS)

print(f'\nMG configs loaded: {list(mg_data.keys())}')
print(f'Skin configs loaded: {list(skin_data.keys())}')

## 3. Figure A: 5-Metric Radar/Bar Chart (Benchmark Style)

参考 benchmark 文章常用的 grouped bar chart，对比 Baseline / PPI / Adaptive PPI 在 5 个 metric 上的表现。

In [ ]:
def plot_5metric_comparison(data, dataset_name, configs=CONFIGS, labels=CONFIG_LABELS, colors=CONFIG_COLORS):
    """Grouped bar chart: 5 metrics × N configs, averaged across slices."""
    metrics_list = ['PCC', 'SSIM', 'SPCC', 'MoransI']  # CMD is lower-is-better, plot separately
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [3, 1]})
    
    # --- Panel A: Higher-is-better metrics ---
    ax = axes[0]
    n_metrics = len(metrics_list)
    n_configs = len([c for c in configs if c in data])
    x = np.arange(n_metrics)
    width = 0.8 / n_configs
    
    for i, cfg in enumerate(configs):
        if cfg not in data:
            continue
        m = data[cfg]['metrics']
        vals = []
        for metric in metrics_list:
            v1 = m['slice1'].get(metric, 0)
            v2 = m['slice2'].get(metric, 0)
            vals.append((v1 + v2) / 2)
        
        offset = (i - n_configs / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=labels[i], color=colors[i],
                      edgecolor='white', linewidth=0.5)
        # Add value labels
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_list, fontsize=11)
    ax.set_ylabel('Score (higher is better ↑)', fontsize=10)
    ax.set_title(f'{dataset_name} — Metric Comparison', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, frameon=False)
    ax.set_ylim(0, 1.05)
    
    # --- Panel B: CMD (lower-is-better) ---
    ax = axes[1]
    cmd_vals = []
    valid_labels = []
    valid_colors = []
    for i, cfg in enumerate(configs):
        if cfg not in data:
            continue
        m = data[cfg]['metrics']
        v1 = m['slice1'].get('CMD', 0)
        v2 = m['slice2'].get('CMD', 0)
        cmd_vals.append((v1 + v2) / 2)
        valid_labels.append(labels[i])
        valid_colors.append(colors[i])
    
    bars = ax.bar(range(len(cmd_vals)), cmd_vals, color=valid_colors, edgecolor='white')
    for bar, v in zip(bars, cmd_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(cmd_vals)))
    ax.set_xticklabels(valid_labels, fontsize=9, rotation=15)
    ax.set_ylabel('CMD (lower is better ↓)', fontsize=10)
    ax.set_title('CMD', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    return fig

fig_mg = plot_5metric_comparison(mg_data, 'Breast Cancer (MG)')
fig_mg.savefig(os.path.join(FIGDIR, 'metric_comparison_MG.pdf'))
fig_mg.savefig(os.path.join(FIGDIR, 'metric_comparison_MG.png'))
plt.show()

fig_skin = plot_5metric_comparison(skin_data, 'Skin Melanoma')
fig_skin.savefig(os.path.join(FIGDIR, 'metric_comparison_skin.pdf'))
fig_skin.savefig(os.path.join(FIGDIR, 'metric_comparison_skin.png'))
plt.show()

## 4. Figure B: Stratified PCC Analysis (Morph-Informativeness)

按 baseline per-gene PCC 将基因分为 3 组，展示 PPI 和 Adaptive PPI 对不同组的差异化贡献。  
**这是回答 Prof Yuan 核心问题的关键图。**

In [ ]:
def plot_stratified_improvement(data, dataset_name, configs=CONFIGS, labels=CONFIG_LABELS, colors=CONFIG_COLORS):
    """3-group stratified PCC bar chart + scatter."""
    if '0_baseline' not in data:
        print('No baseline data, skip'); return None
    
    # Average PCC across slices for baseline
    bl_pcc_s1 = data['0_baseline'].get('pcc_s1', np.array([]))
    bl_pcc_s2 = data['0_baseline'].get('pcc_s2', np.array([]))
    if len(bl_pcc_s1) == 0:
        print('No per-gene PCC arrays, skip'); return None
    bl_pcc = np.nan_to_num((bl_pcc_s1 + bl_pcc_s2) / 2)
    n_genes = len(bl_pcc)
    
    # Stratify
    groups = {
        'Morph-Uninf\n(PCC<0.15)': bl_pcc < 0.15,
        'Morph-Partial\n(0.15-0.30)': (bl_pcc >= 0.15) & (bl_pcc < 0.30),
        'Morph-Inf\n(PCC≥0.30)': bl_pcc >= 0.30,
    }
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # --- Panel A: Grouped bar chart per group ---
    ax = axes[0]
    group_names = list(groups.keys())
    n_groups = len(group_names)
    n_cfgs = len([c for c in configs if c in data and 'pcc_s1' in data[c]])
    x = np.arange(n_groups)
    width = 0.8 / max(n_cfgs, 1)
    
    ci = 0
    for j, cfg in enumerate(configs):
        if cfg not in data or 'pcc_s1' not in data[cfg]:
            continue
        pcc_avg = np.nan_to_num((data[cfg]['pcc_s1'] + data[cfg]['pcc_s2']) / 2)
        vals = [np.mean(pcc_avg[mask]) if mask.sum() > 0 else 0 for _, mask in groups.items()]
        offset = (ci - n_cfgs / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=labels[j], color=colors[j], edgecolor='white')
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)
        ci += 1
    
    ax.set_xticks(x)
    ax.set_xticklabels(group_names, fontsize=9)
    ax.set_ylabel('Mean PCC', fontsize=10)
    ax.set_title(f'{dataset_name}\nPCC by Morphology Group', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    
    # Count labels
    for i, (gname, mask) in enumerate(groups.items()):
        ax.text(i, -0.02, f'n={mask.sum()}', ha='center', fontsize=8, color='gray', transform=ax.get_xaxis_transform())
    
    # --- Panel B: Scatter — baseline PCC vs ΔPCC ---
    ax = axes[1]
    for j, cfg in enumerate(configs[1:], 1):  # skip baseline
        if cfg not in data or 'pcc_s1' not in data[cfg]:
            continue
        pcc_avg = np.nan_to_num((data[cfg]['pcc_s1'] + data[cfg]['pcc_s2']) / 2)
        delta = pcc_avg - bl_pcc
        ax.scatter(bl_pcc, delta, s=8, alpha=0.5, color=colors[j], label=labels[j], edgecolors='none')
    
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Baseline PCC', fontsize=10)
    ax.set_ylabel('ΔPCC (method − baseline)', fontsize=10)
    ax.set_title('Per-Gene Improvement', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    
    # --- Panel C: ΔPCC distribution (violin) ---
    ax = axes[2]
    violin_data = []
    violin_labels = []
    violin_colors_used = []
    for j, cfg in enumerate(configs[1:], 1):
        if cfg not in data or 'pcc_s1' not in data[cfg]:
            continue
        pcc_avg = np.nan_to_num((data[cfg]['pcc_s1'] + data[cfg]['pcc_s2']) / 2)
        delta = pcc_avg - bl_pcc
        violin_data.append(delta)
        violin_labels.append(labels[j])
        violin_colors_used.append(colors[j])
    
    if violin_data:
        parts = ax.violinplot(violin_data, positions=range(len(violin_data)), showmeans=True, showmedians=True)
        for i, pc in enumerate(parts['bodies']):
            pc.set_facecolor(violin_colors_used[i])
            pc.set_alpha(0.7)
        ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
        ax.set_xticks(range(len(violin_data)))
        ax.set_xticklabels(violin_labels, fontsize=9)
        ax.set_ylabel('ΔPCC distribution', fontsize=10)
        ax.set_title('ΔPCC Distribution', fontsize=11, fontweight='bold')
        
        # Add stats
        for i, d in enumerate(violin_data):
            mean_d = np.mean(d)
            pct_pos = (d > 0).mean() * 100
            ax.text(i, ax.get_ylim()[1] * 0.9, f'mean={mean_d:+.4f}\n{pct_pos:.0f}% genes↑',
                    ha='center', fontsize=7, color=violin_colors_used[i])
    
    plt.tight_layout()
    return fig

fig_strat_mg = plot_stratified_improvement(mg_data, 'Breast Cancer (MG)')
if fig_strat_mg:
    fig_strat_mg.savefig(os.path.join(FIGDIR, 'stratified_pcc_MG.pdf'))
    fig_strat_mg.savefig(os.path.join(FIGDIR, 'stratified_pcc_MG.png'))
plt.show()

fig_strat_skin = plot_stratified_improvement(skin_data, 'Skin Melanoma')
if fig_strat_skin:
    fig_strat_skin.savefig(os.path.join(FIGDIR, 'stratified_pcc_skin.pdf'))
    fig_strat_skin.savefig(os.path.join(FIGDIR, 'stratified_pcc_skin.png'))
plt.show()

## 5. Figure C: Gene-Adaptive Gate (γ) Analysis

v2 新增的关键可视化：展示每个基因学到的 PPI 依赖度 γ_g。  
γ≈0 → 依赖形态学预测，γ≈1 → 依赖 PPI 邻居信息。

In [ ]:
def plot_gate_analysis(data, dataset_name):
    """Gate γ vs baseline PCC scatter + distribution."""
    if '4_adaptive_ppi' not in data:
        print('No adaptive_ppi data'); return None
    if 'gate_s1' not in data['4_adaptive_ppi']:
        print('No gate values saved'); return None
    
    gate_s1 = data['4_adaptive_ppi']['gate_s1']
    gate_s2 = data['4_adaptive_ppi'].get('gate_s2', gate_s1)
    gate_avg = (gate_s1 + gate_s2) / 2
    
    bl_pcc_s1 = data['0_baseline'].get('pcc_s1', np.zeros_like(gate_s1))
    bl_pcc_s2 = data['0_baseline'].get('pcc_s2', np.zeros_like(gate_s1))
    bl_pcc = np.nan_to_num((bl_pcc_s1 + bl_pcc_s2) / 2)
    
    n_genes = min(len(gate_avg), len(bl_pcc))
    gate_avg = gate_avg[:n_genes]
    bl_pcc = bl_pcc[:n_genes]
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # --- Panel A: Gate vs Baseline PCC scatter ---
    ax = axes[0]
    sc = ax.scatter(bl_pcc, gate_avg, s=15, c=gate_avg, cmap='RdYlBu_r',
                    vmin=0, vmax=1, edgecolors='black', linewidth=0.3, alpha=0.8)
    plt.colorbar(sc, ax=ax, label='Gate γ', shrink=0.8)
    ax.set_xlabel('Baseline PCC (morphology informativeness)', fontsize=10)
    ax.set_ylabel('Learned Gate γ (PPI dependence)', fontsize=10)
    ax.set_title(f'{dataset_name}\nGate γ vs Baseline PCC', fontsize=11, fontweight='bold')
    
    # Trend line
    z = np.polyfit(bl_pcc, gate_avg, 1)
    p = np.poly1d(z)
    x_line = np.linspace(bl_pcc.min(), bl_pcc.max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=1.5, alpha=0.7)
    r, pval = pearsonr(bl_pcc, gate_avg)
    ax.text(0.05, 0.95, f'r={r:.3f}, p={pval:.1e}', transform=ax.transAxes,
            fontsize=9, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # --- Panel B: Gate distribution ---
    ax = axes[1]
    ax.hist(gate_avg, bins=40, color='#E64B35', edgecolor='white', alpha=0.8)
    ax.axvline(0.5, color='black', linewidth=1, linestyle='--', label='γ=0.5')
    ax.set_xlabel('Gate γ', fontsize=10)
    ax.set_ylabel('Number of genes', fontsize=10)
    ax.set_title('Gate Distribution', fontsize=11, fontweight='bold')
    n_high = (gate_avg > 0.5).sum()
    n_low = (gate_avg < 0.5).sum()
    ax.text(0.05, 0.95, f'PPI-dependent (γ>0.5): {n_high}\nMorph-dependent (γ<0.5): {n_low}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
    ax.legend(fontsize=8)
    
    # --- Panel C: Top PPI-dependent & morph-dependent genes ---
    ax = axes[2]
    top_k = 10
    top_ppi_idx = np.argsort(gate_avg)[-top_k:][::-1]
    top_morph_idx = np.argsort(gate_avg)[:top_k]
    
    # Load gate summary for gene names
    gate_summary = data['4_adaptive_ppi'].get('gate_summary_s1', {})
    top_ppi_genes = gate_summary.get('top_ppi_dependent', [])
    top_morph_genes = gate_summary.get('top_morph_dependent', [])
    
    all_genes = []
    all_vals = []
    all_colors_bar = []
    
    for g in top_morph_genes[:5]:
        all_genes.append(g['gene'])
        all_vals.append(g['gate'])
        all_colors_bar.append('#4DBBD5')
    all_genes.append('---')
    all_vals.append(0)
    all_colors_bar.append('white')
    for g in reversed(top_ppi_genes[:5]):
        all_genes.append(g['gene'])
        all_vals.append(g['gate'])
        all_colors_bar.append('#E64B35')
    
    y_pos = range(len(all_genes))
    ax.barh(y_pos, all_vals, color=all_colors_bar, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(all_genes, fontsize=8)
    ax.set_xlabel('Gate γ', fontsize=10)
    ax.set_title('Top Genes by PPI Dependence', fontsize=11, fontweight='bold')
    ax.legend(handles=[
        Patch(facecolor='#4DBBD5', label='Morph-dependent (low γ)'),
        Patch(facecolor='#E64B35', label='PPI-dependent (high γ)'),
    ], fontsize=8, frameon=False, loc='lower right')
    
    plt.tight_layout()
    return fig

fig_gate_mg = plot_gate_analysis(mg_data, 'Breast Cancer (MG)')
if fig_gate_mg:
    fig_gate_mg.savefig(os.path.join(FIGDIR, 'gate_analysis_MG.pdf'))
    fig_gate_mg.savefig(os.path.join(FIGDIR, 'gate_analysis_MG.png'))
plt.show()

fig_gate_skin = plot_gate_analysis(skin_data, 'Skin Melanoma')
if fig_gate_skin:
    fig_gate_skin.savefig(os.path.join(FIGDIR, 'gate_analysis_skin.pdf'))
    fig_gate_skin.savefig(os.path.join(FIGDIR, 'gate_analysis_skin.png'))
plt.show()

## 6. Figure D: Top Improved & Degraded Genes (3-way)

In [ ]:
def plot_top_genes_3way(data, dataset_name, top_k=15):
    """Top improved/degraded genes: Adaptive PPI vs Baseline."""
    if '4_adaptive_ppi' not in data or '0_baseline' not in data:
        print('Missing data'); return None
    if 'pcc_s1' not in data['0_baseline'] or 'pcc_s1' not in data['4_adaptive_ppi']:
        print('Missing per-gene PCC'); return None
    
    bl_pcc = np.nan_to_num((data['0_baseline']['pcc_s1'] + data['0_baseline']['pcc_s2']) / 2)
    adp_pcc = np.nan_to_num((data['4_adaptive_ppi']['pcc_s1'] + data['4_adaptive_ppi']['pcc_s2']) / 2)
    delta = adp_pcc - bl_pcc
    n_genes = len(delta)
    gene_idx = np.arange(n_genes)
    
    # Also load PPI v1 if available
    has_v1 = '3_all_three' in data and 'pcc_s1' in data['3_all_three']
    if has_v1:
        v1_pcc = np.nan_to_num((data['3_all_three']['pcc_s1'] + data['3_all_three']['pcc_s2']) / 2)
        delta_v1 = v1_pcc - bl_pcc
    
    # Top improved
    top_idx = np.argsort(delta)[-top_k:][::-1]
    bot_idx = np.argsort(delta)[:top_k]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top improved
    ax = axes[0]
    y = np.arange(top_k)
    ax.barh(y - 0.15, delta[top_idx], 0.3, color='#E64B35', label='v2 Adaptive', edgecolor='white')
    if has_v1:
        ax.barh(y + 0.15, delta_v1[top_idx], 0.3, color='#4DBBD5', label='v1 PPI', edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels([f'Gene {i} (bl={bl_pcc[i]:.2f})' for i in top_idx], fontsize=8)
    ax.set_xlabel('ΔPCC vs Baseline', fontsize=10)
    ax.set_title(f'Top {top_k} Improved', fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(fontsize=8, frameon=False)
    
    # Top degraded
    ax = axes[1]
    ax.barh(y - 0.15, delta[bot_idx], 0.3, color='#E64B35', label='v2 Adaptive', edgecolor='white')
    if has_v1:
        ax.barh(y + 0.15, delta_v1[bot_idx], 0.3, color='#4DBBD5', label='v1 PPI', edgecolor='white')
    ax.set_yticks(y)
    ax.set_yticklabels([f'Gene {i} (bl={bl_pcc[i]:.2f})' for i in bot_idx], fontsize=8)
    ax.set_xlabel('ΔPCC vs Baseline', fontsize=10)
    ax.set_title(f'Top {top_k} Degraded', fontsize=11, fontweight='bold')
    ax.invert_yaxis()
    ax.legend(fontsize=8, frameon=False)
    
    fig.suptitle(f'{dataset_name} — Gene-Level PCC Change', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    return fig

fig_top_mg = plot_top_genes_3way(mg_data, 'Breast Cancer (MG)')
if fig_top_mg:
    fig_top_mg.savefig(os.path.join(FIGDIR, 'top_genes_MG.pdf'))
plt.show()

fig_top_skin = plot_top_genes_3way(skin_data, 'Skin Melanoma')
if fig_top_skin:
    fig_top_skin.savefig(os.path.join(FIGDIR, 'top_genes_skin.pdf'))
plt.show()

## 7. Summary Table

生成最终的 metrics 对比表（Markdown + CSV）。

In [ ]:
def make_summary_table(data, dataset_name, configs=CONFIGS, labels=CONFIG_LABELS):
    """Print and return summary DataFrame."""
    rows = []
    for cfg, label in zip(configs, labels):
        if cfg not in data:
            continue
        m = data[cfg]['metrics']
        s1, s2 = m['slice1'], m['slice2']
        row = {
            'Method': label,
            'PCC': round((s1.get('PCC',0) + s2.get('PCC',0)) / 2, 4),
            'SSIM': round((s1.get('SSIM',0) + s2.get('SSIM',0)) / 2, 4),
            'CMD↓': round((s1.get('CMD',0) + s2.get('CMD',0)) / 2, 4),
            'MoransI': round((s1.get('MoransI',0) + s2.get('MoransI',0)) / 2, 4),
            'SPCC': round((s1.get('SPCC',0) + s2.get('SPCC',0)) / 2, 4),
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    print(f'\n{dataset_name}')
    print(df.to_markdown(index=False))
    return df

df_mg = make_summary_table(mg_data, 'Breast Cancer (MG)')
df_skin = make_summary_table(skin_data, 'Skin Melanoma')

# Save CSV
if df_mg is not None:
    df_mg.to_csv(os.path.join(FIGDIR, 'summary_MG.csv'), index=False)
if df_skin is not None:
    df_skin.to_csv(os.path.join(FIGDIR, 'summary_skin.csv'), index=False)
print('\nCSVs saved.')

## 8. Stratified Summary Table

按 morphology-informativeness 分组的指标表，直接回应 Prof Yuan 的问题。

In [ ]:
def make_stratified_table(data, dataset_name, configs=CONFIGS, labels=CONFIG_LABELS):
    """Per morphology group PCC summary."""
    if '0_baseline' not in data or 'pcc_s1' not in data['0_baseline']:
        print('No baseline per-gene data'); return None
    
    bl_pcc = np.nan_to_num((data['0_baseline']['pcc_s1'] + data['0_baseline']['pcc_s2']) / 2)
    groups = {
        'Morph-Uninf (PCC<0.15)': bl_pcc < 0.15,
        'Morph-Partial (0.15-0.30)': (bl_pcc >= 0.15) & (bl_pcc < 0.30),
        'Morph-Inf (PCC≥0.30)': bl_pcc >= 0.30,
        'Overall': np.ones(len(bl_pcc), dtype=bool),
    }
    
    rows = []
    for gname, mask in groups.items():
        n = mask.sum()
        row = {'Group': gname, 'N_genes': n}
        for cfg, label in zip(configs, labels):
            if cfg not in data or 'pcc_s1' not in data[cfg]:
                continue
            pcc_avg = np.nan_to_num((data[cfg]['pcc_s1'] + data[cfg]['pcc_s2']) / 2)
            row[f'PCC_{label}'] = round(float(np.mean(pcc_avg[mask])), 4) if n > 0 else 0
        rows.append(row)
    
    df = pd.DataFrame(rows)
    print(f'\n{dataset_name} — Stratified PCC')
    print(df.to_markdown(index=False))
    return df

df_strat_mg = make_stratified_table(mg_data, 'Breast Cancer (MG)')
df_strat_skin = make_stratified_table(skin_data, 'Skin Melanoma')

if df_strat_mg is not None:
    df_strat_mg.to_csv(os.path.join(FIGDIR, 'stratified_MG.csv'), index=False)
if df_strat_skin is not None:
    df_strat_skin.to_csv(os.path.join(FIGDIR, 'stratified_skin.csv'), index=False)

## 9. Statistical Significance (Wilcoxon)

In [ ]:
def wilcoxon_test(data, dataset_name):
    """Wilcoxon signed-rank test: each method vs baseline."""
    if '0_baseline' not in data or 'pcc_s1' not in data['0_baseline']:
        return
    bl_pcc = np.nan_to_num((data['0_baseline']['pcc_s1'] + data['0_baseline']['pcc_s2']) / 2)
    
    print(f'\n{dataset_name} — Wilcoxon signed-rank test vs Baseline')
    print(f'{"Method":<25} {"Mean ΔPCC":>10} {"% genes ↑":>10} {"p-value":>12} {"Sig":>8}')
    print('-' * 70)
    
    for cfg, label in zip(CONFIGS[1:], CONFIG_LABELS[1:]):
        if cfg not in data or 'pcc_s1' not in data[cfg]:
            continue
        pcc = np.nan_to_num((data[cfg]['pcc_s1'] + data[cfg]['pcc_s2']) / 2)
        n = min(len(bl_pcc), len(pcc))
        delta = pcc[:n] - bl_pcc[:n]
        mask = ~np.isnan(delta)
        if mask.sum() < 10:
            continue
        stat, pval = wilcoxon(pcc[:n][mask], bl_pcc[:n][mask])
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'n.s.'
        mean_d = np.mean(delta[mask])
        pct = (delta[mask] > 0).mean() * 100
        print(f'{label:<25} {mean_d:>+10.5f} {pct:>9.1f}% {pval:>12.2e} {sig:>8}')

wilcoxon_test(mg_data, 'Breast Cancer (MG)')
wilcoxon_test(skin_data, 'Skin Melanoma')